# 卷积神经网络（上）卷积运算 + LeNet 手写数字识别

本节做三件事：

1. 从最朴素的 `for` 循环写一个二维卷积，对照 `nn.Conv2d` 输出验证一致。
2. 用手设计的 kernel 在真实图像上做**边缘检测**，直观感受卷积。
3. 实现 **LeNet-5** 在 MNIST 子集上做手写数字识别。

## 1. 二维卷积：手写 vs `nn.Conv2d`

信号处理里 "convolution" 严格意义上 kernel 要翻转，而深度学习里习惯说的 "convolution" 其实是**互相关（cross-correlation）**——kernel 不翻转，直接滑动相乘相加。`nn.Conv2d` 就是后者。

下面对单通道输入手写一遍：

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib.pyplot as plt

torch.manual_seed(0)

def conv2d_naive(X, W, b=0.0, stride=1, padding=0):
    """X: [H, W], W: [kH, kW]; returns [H_out, W_out]. 仅做单通道、无 batch。"""
    if padding:
        X = F.pad(X, (padding, padding, padding, padding))
    H, Wd = X.shape; kH, kW = W.shape
    H_out = (H - kH) // stride + 1
    W_out = (Wd - kW) // stride + 1
    out = torch.zeros(H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            h = i * stride; w = j * stride
            out[i, j] = (X[h:h+kH, w:w+kW] * W).sum() + b
    return out

x = torch.arange(25, dtype=torch.float32).reshape(5, 5)
k = torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]])
manual = conv2d_naive(x, k, stride=1, padding=0)
print('naive output:'); print(manual)

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from nndl.runner import RunnerV3


In [ ]:
# nn.Conv2d 的等价：[N, C_in, H, W] -> [N, C_out, H_out, W_out]
X = x.reshape(1, 1, 5, 5)
conv = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0, bias=False)
with torch.no_grad():
    conv.weight.copy_(k.reshape(1, 1, 3, 3))
out_torch = conv(X).squeeze()
print('nn.Conv2d output:'); print(out_torch)
print('max abs diff:', (out_torch - manual).abs().max().item())

### 形状公式

$H_\text{out} = \lfloor (H + 2p - k) / s \rfloor + 1$，宽度同理。下面用 `nn.Conv2d` 直接验证 padding/stride 组合：

In [ ]:
X = torch.zeros(1, 1, 7, 7)
for k_size, pad, st in [(3, 0, 1), (3, 1, 1), (3, 1, 2), (5, 2, 1)]:
    c = nn.Conv2d(1, 1, kernel_size=k_size, padding=pad, stride=st)
    out = c(X)
    print(f'kernel={k_size} pad={pad} stride={st} -> {tuple(out.shape)}')

## 2. 边缘检测：手设计 kernel 在真实图像上

拉普拉斯算子 $\begin{bmatrix}0 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 0\end{bmatrix}$ 高亮像素与邻域的差异——即边缘。

In [ ]:
import os
from torchvision import datasets, transforms

CACHE = os.path.expanduser('~/.cache/torch_data')
mnist = datasets.MNIST(CACHE, train=True, download=True, transform=transforms.ToTensor())
img, label = mnist[0]      # [1, 28, 28]

laplacian = torch.tensor([[0., -1, 0], [-1, 4, -1], [0., -1, 0]]).reshape(1, 1, 3, 3)
edges = F.conv2d(img.unsqueeze(0), laplacian, padding=1).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img.squeeze(), cmap='gray'); axes[0].set_title(f'original (label={label})')
axes[1].imshow(edges,         cmap='gray'); axes[1].set_title('Laplacian edges')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## 3. LeNet-5

经典结构（输入 1×32×32 灰度图）：

```
Conv2d(1→6, k=5)  -> ReLU -> MaxPool2d(2)   # 32×32 -> 28×28 -> 14×14
Conv2d(6→16, k=5) -> ReLU -> MaxPool2d(2)   # 14×14 -> 10×10 -> 5×5
Linear(16*5*5 → 120) -> ReLU
Linear(120 → 84) -> ReLU
Linear(84 → 10)
```

MNIST 是 28×28，所以输入做 padding 到 32×32（或者把第一个 Conv 改成 padding=2）。下面用 padding=2 的方案。
> **关于激活函数**：1998 年原版 LeNet-5 用的是 tanh / sigmoid。这里改用 ReLU——它在深网络上训练更快、更不容易梯度消失，是现代 CNN 的默认选择。结构其他部分保持原样。

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, n_class=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)   # 28x28 -> 28x28
        self.pool1 = nn.MaxPool2d(2, 2)                          # -> 14x14
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)             # -> 10x10
        self.pool2 = nn.MaxPool2d(2, 2)                          # -> 5x5
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,  n_class)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# 看每层输出形状：显式 forward 一步一步走，不依赖 __init__ 属性顺序
net = LeNet5()
x = torch.zeros(1, 1, 28, 28)
print(f'input : {tuple(x.shape)}')
x = net.conv1(x); print(f'conv1 : {tuple(x.shape)}')
x = net.pool1(x); print(f'pool1 : {tuple(x.shape)}')
x = net.conv2(x); print(f'conv2 : {tuple(x.shape)}')
x = net.pool2(x); print(f'pool2 : {tuple(x.shape)}')
x = x.flatten(start_dim=1); print(f'flat  : {tuple(x.shape)}')
x = net.fc1(x);   print(f'fc1   : {tuple(x.shape)}')
x = net.fc2(x);   print(f'fc2   : {tuple(x.shape)}')
x = net.fc3(x);   print(f'fc3   : {tuple(x.shape)}')

### 参数量

In [ ]:
total = sum(p.numel() for p in LeNet5().parameters())
print(f'total params: {total:,}')
for name, p in LeNet5().named_parameters():
    print(f'  {name:18s} {str(tuple(p.shape)):20s} {p.numel():>8,d}')

## 4. 在 MNIST 子集上训练

为了 notebook 跑得快，只用 5000 训练 + 1000 测试。完整 60000/10000 在更大 epoch 下能轻松到 99% 以上。

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST(CACHE, train=True,  download=True, transform=transform)
test_full  = datasets.MNIST(CACHE, train=False, download=True, transform=transform)

torch.manual_seed(0)
tr_idx = torch.randperm(len(train_full))[:5000]
te_idx = torch.randperm(len(test_full))[:1000]
train_set = Subset(train_full, tr_idx.tolist())
test_set  = Subset(test_full,  te_idx.tolist())
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=128)
print(f'train {len(train_set)}  test {len(test_set)}')

In [ ]:
torch.manual_seed(0)
model = LeNet5()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

from nndl import accuracy

runner = RunnerV3(model, optimizer, loss_fn, metric_fn=accuracy, higher_is_better=True)
runner.fit(train_loader, test_loader, num_epochs=5, log_every=1)

### 看几个预测

In [ ]:
model.eval()
x, y = next(iter(test_loader))
with torch.no_grad():
    probs = F.softmax(model(x[:8]), dim=1)
preds = probs.argmax(dim=1)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, img, true, p, pr in zip(axes, x[:8], y[:8], preds, probs):
    ax.imshow(img.squeeze(), cmap='gray')
    ok = '✓' if true == p else '✗'
    ax.set_title(f'{ok} pred={p.item()}({pr[p].item():.2f}) true={true.item()}', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

### 看第一个 conv 层学到的 kernel

6 个 5×5 的 filter。训练前是随机噪声，训练后会形成边缘、纹理之类的探测器。

In [ ]:
kernels = model.conv1.weight.detach().squeeze(1)   # [6, 5, 5]
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, k in zip(axes, kernels):
    ax.imshow(k, cmap='RdBu_r'); ax.axis('off')
plt.suptitle('conv1 learned kernels'); plt.show()

## 小结

- 深度学习里的 "conv" 实际是 cross-correlation，`nn.Conv2d` 即是该操作。
- 形状公式：$H_\text{out} = \lfloor (H + 2p - k) / s \rfloor + 1$；用 padding=k//2 + stride=1 可以保持空间尺寸不变。
- LeNet-5 在 MNIST 5000 训练子集上 5 个 epoch 能到 ~95% 测试准确率；完整 60k 训练 + 更多 epoch 可以到 99%。
- 第一卷积层的 kernel 在训练后会变成边缘 / 纹理探测器，可直接 imshow 看。
- chap5 下做 ResNet：用残差连接训更深的网络，在 CIFAR-10 上看效果。

## 附：从零实现卷积与汇聚算子族（对照书本）

前面用的是 `nn.Conv2d`，这一节换个视角：**完全用 `for` 循环和张量切片，从零把卷积/汇聚算子写出来**，与配套书《案例与实践（2）》§5.1～§5.2 一一对应。目的是看清卷积“滑窗—相乘—求和”的机械过程，以及多通道是怎么逐通道叠加的。

> 这些朴素实现只为讲清原理，速度比 `nn.Conv2d` 慢几个数量级（底层是 im2col + GEMM）。真做项目仍用 `nn.Conv2d` / `F.conv2d`。

本节按四步推进：① 最朴素的二维卷积 → ② 加入步长与零填充 → ③ 平均/最大汇聚 → ④ 多输入多输出通道的四步分解；最后把它们拼成一个 LeNet-5，逐层验证特征图形状。

### 1. 最朴素的二维卷积算子

按公式 $y_{ij}=\sum_{u}\sum_{v} w_{uv}\,x_{i+u,\,j+v}$ 直接双重循环。输入约定为 `[B, M, N]`（B 是样本数，单通道），卷积核 `[U, V]`，输出 `[B, M-U+1, N-V+1]`。

In [ ]:
import torch
import torch.nn as nn


class Conv2dNaive(nn.Module):
    """最朴素的单通道二维卷积（互相关），仅用循环+切片。"""
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(kernel_size, kernel_size, dtype=torch.float32))

    def forward(self, X):
        """输入 X：[B, M, N]；输出：[B, M-U+1, N-V+1]"""
        u, v = self.weight.shape
        output = torch.zeros([X.shape[0], X.shape[1] - u + 1, X.shape[2] - v + 1])
        for i in range(output.shape[1]):
            for j in range(output.shape[2]):
                output[:, i, j] = torch.sum(X[:, i:i + u, j:j + v] * self.weight, dim=(1, 2))
        return output


# 随机构造一个二维输入矩阵
torch.manual_seed(100)
inputs = torch.randn(size=[2, 3, 3])
conv2d = Conv2dNaive(kernel_size=2)
outputs = conv2d(inputs)
print(f"input shape : {tuple(inputs.shape)}")
print(f"output shape: {tuple(outputs.shape)}")
print(f"output:\n{outputs}")

### 2. 带步长和零填充的二维卷积算子

两个常用变种：
- **步长 stride**：卷积核每次滑动跨越的元素数，$S>1$ 时对输出做下采样、压缩特征图。
- **零填充 padding**：先在输入边缘补 $P$ 圈 0 再卷积，让输出尺寸不必随卷积核变大而迅速缩水。

输出尺寸公式：$M'=\lfloor (M+2P-U)/S\rfloor+1$，宽度同理。下面把步长与零填充一起塞进算子，并验证 `kernel=3, padding=1` 时 `stride=1` 保持尺寸、`stride=2` 宽高减半。

In [ ]:
class Conv2dStridePad(nn.Module):
    """带步长与零填充的单通道二维卷积。"""
    def __init__(self, kernel_size, stride=1, padding=0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(kernel_size, kernel_size, dtype=torch.float32))
        self.stride = stride      # 步长
        self.padding = padding    # 零填充

    def forward(self, X):
        # 零填充：在四周补 padding 圈 0
        new_X = torch.zeros([X.shape[0], X.shape[1] + 2 * self.padding, X.shape[2] + 2 * self.padding])
        new_X[:, self.padding:X.shape[1] + self.padding, self.padding:X.shape[2] + self.padding] = X
        u, v = self.weight.shape
        output_w = (new_X.shape[1] - u) // self.stride + 1
        output_h = (new_X.shape[2] - v) // self.stride + 1
        output = torch.zeros([X.shape[0], output_w, output_h])
        for i in range(output.shape[1]):
            for j in range(output.shape[2]):
                output[:, i, j] = torch.sum(
                    new_X[:, self.stride * i:self.stride * i + u, self.stride * j:self.stride * j + v] * self.weight,
                    dim=(1, 2))
        return output


inputs = torch.randn(size=[2, 8, 8])
conv2d_padding = Conv2dStridePad(kernel_size=3, padding=1)
outputs = conv2d_padding(inputs)
print(f"kernel=3 padding=1 stride=1: {tuple(inputs.shape)} -> {tuple(outputs.shape)}")
conv2d_stride = Conv2dStridePad(kernel_size=3, stride=2, padding=1)
outputs = conv2d_stride(inputs)
print(f"kernel=3 padding=1 stride=2: {tuple(inputs.shape)} -> {tuple(outputs.shape)}")

### 3. 汇聚层算子（平均/最大）

**汇聚（Pooling）** 对特征图做下采样：把局部窗口内的多个响应压成一个，减少特征数量、带来一定平移不变性。两种常见模式：
- **最大汇聚**：取窗口内最强响应，保留显著特征更激进；
- **平均汇聚**：取窗口内均值，平滑性更好。

输入约定 `[B, C, H, W]`。

In [ ]:
class Pool2D(nn.Module):
    """支持 max / avg 两种模式的简易汇聚算子。"""
    def __init__(self, size=(2, 2), mode='max', stride=1):
        super().__init__()
        self.mode = mode          # 汇聚方式：'max' 或 'avg'
        self.h, self.w = size
        self.stride = stride

    def forward(self, x):
        output_w = (x.shape[2] - self.w) // self.stride + 1
        output_h = (x.shape[3] - self.h) // self.stride + 1
        output = x.new_zeros(x.shape[0], x.shape[1], output_w, output_h)
        for i in range(output.shape[2]):
            for j in range(output.shape[3]):
                window = x[:, :, self.stride * i:self.stride * i + self.w, self.stride * j:self.stride * j + self.h]
                if self.mode == 'max':
                    output[:, :, i, j] = torch.amax(window, dim=(2, 3))
                elif self.mode == 'avg':
                    output[:, :, i, j] = torch.mean(window, dim=(2, 3))
        return output


feat = torch.randn(2, 4, 8, 8)
print(f"input            : {tuple(feat.shape)}")
print(f"max  pool 2x2 s2 : {tuple(Pool2D((2, 2), 'max', 2)(feat).shape)}")
print(f"avg  pool 2x2 s2 : {tuple(Pool2D((2, 2), 'avg', 2)(feat).shape)}")

动手练习：构造一个 $4\times4$ 特征图，把 $2\times2$、步长为 2 的平均汇聚和最大汇聚过程画出来。

In [ ]:
# 可视化平均汇聚与最大汇聚的窗口和输出
feature_map = torch.tensor([
    [1., 2., 5., 0.],
    [3., 4., 2., 1.],
    [0., 1., 6., 2.],
    [2., 3., 1., 4.],
])
x = feature_map.reshape(1, 1, 4, 4)
avg_out = Pool2D((2, 2), mode='avg', stride=2)(x)[0, 0]
max_out = Pool2D((2, 2), mode='max', stride=2)(x)[0, 0]


def draw_matrix(ax, mat, title, mark_windows=False):
    data = mat.detach().cpu().numpy()
    ax.imshow(data, cmap='Blues', vmin=0, vmax=6)
    ax.set_title(title)
    ax.set_xticks(range(data.shape[1]))
    ax.set_yticks(range(data.shape[0]))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xticks([i - 0.5 for i in range(data.shape[1] + 1)], minor=True)
    ax.set_yticks([i - 0.5 for i in range(data.shape[0] + 1)], minor=True)
    ax.grid(which='minor', color='white', linewidth=2)
    ax.tick_params(which='both', length=0)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i, j]:g}', ha='center', va='center', color='black', fontsize=12)

    if mark_windows:
        colors = ['#d62728', '#2ca02c', '#ff7f0e', '#9467bd']
        k = 0
        for r in range(0, data.shape[0], 2):
            for c in range(0, data.shape[1], 2):
                ax.add_patch(plt.Rectangle((c - 0.48, r - 0.48), 1.96, 1.96,
                                           fill=False, edgecolor=colors[k], linewidth=3))
                k += 1


fig, axes = plt.subplots(1, 3, figsize=(9, 3))
draw_matrix(axes[0], feature_map, 'input: 4x4', mark_windows=True)
draw_matrix(axes[1], avg_out, 'avg pool: 2x2, stride=2')
draw_matrix(axes[2], max_out, 'max pool: 2x2, stride=2')
plt.tight_layout()
plt.show()

print('avg pooling output:\n', avg_out)
print('max pooling output:\n', max_out)

### 4. 多输入多输出通道的卷积层（四步分解）

真实图像不是单通道：彩色图有 R/G/B 三通道，中间层特征图通道更多。一层里也通常并行多个卷积核以提取多种特征。把多通道卷积拆成四步看：

1. **单通道卷积**：二维输入 $\times$ 二维核 → 二维特征图（即上面的算子）。
2. **多输入→单输出**：$D$ 个输入通道各配一个二维核，分别卷积后逐元素相加，得到 1 张特征图，再加偏置 $b$。
3. **多输入→多输出**：把第 2 步重复 $P$ 次（$P$ 组三维核），堆出 $P$ 张特征图。
4. **非线性激活**：对输出逐元素施加激活（工程上通常单独作为算子，这里不并入）。

卷积核张量布局 `[P, D, U, V]`（输出通道、输入通道、核高、核宽），输入/输出 `[B, D, M, N]`。

In [ ]:
class Conv2dMulti(nn.Module):
    """多输入多输出通道卷积，逐位置滑窗的四步分解实现（教学用，非生产实现）。"""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        # 卷积核：[P, D, U, V]
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size, dtype=torch.float32))
        # 偏置：每个输出通道一个
        self.bias = nn.Parameter(torch.zeros(out_channels, 1, dtype=torch.float32))
        self.stride = stride
        self.padding = padding
        self.in_channels = in_channels
        self.out_channels = out_channels

    # 第(1)步：单通道特征图的卷积
    def single_forward(self, X, weight):
        new_X = X.new_zeros(X.shape[0], X.shape[1] + 2 * self.padding, X.shape[2] + 2 * self.padding)
        new_X[:, self.padding:X.shape[1] + self.padding, self.padding:X.shape[2] + self.padding] = X
        u, v = weight.shape
        output_w = (new_X.shape[1] - u) // self.stride + 1
        output_h = (new_X.shape[2] - v) // self.stride + 1
        output = X.new_zeros(X.shape[0], output_w, output_h)
        for i in range(output.shape[1]):
            for j in range(output.shape[2]):
                output[:, i, j] = torch.sum(
                    new_X[:, self.stride * i:self.stride * i + u, self.stride * j:self.stride * j + v] * weight,
                    dim=(1, 2))
        return output

    # 第(2)步：多输入通道到单输出通道（各通道卷积后逐元素相加，再加偏置）
    def multi2single_forward(self, inputs, weight, b):
        return torch.sum(
            torch.stack([self.single_forward(inputs[:, i, :, :], weight[i]) for i in range(self.in_channels)], dim=1),
            dim=1) + b

    # 第(3)步：多输入通道到多输出通道（重复 P 次后按通道堆叠）
    def multi2multi_forward(self, inputs, weights, bias):
        return torch.stack([self.multi2single_forward(inputs, w, b) for w, b in zip(weights, bias)], dim=1)

    def forward(self, inputs):
        return self.multi2multi_forward(inputs, self.weight, self.bias)


torch.manual_seed(100)
x = torch.randn(2, 3, 8, 8)                       # [B=2, D=3, 8, 8]
conv = Conv2dMulti(in_channels=3, out_channels=4, kernel_size=3, padding=1)
out = conv(x)
print(f"input  [B,D,M,N]: {tuple(x.shape)}")
print(f"output [B,P,M,N]: {tuple(out.shape)}")    # 期望 [2, 4, 8, 8]

### 5. 用从零算子做拉普拉斯边缘检测

边缘检测里常用**拉普拉斯算子**——一个 $3\times3$ 卷积核，中心为 8、四周为 $-1$，对像素邻域的强度突变（也就是边缘）特别敏感。

这里不依赖外部图片，直接用 NumPy **合成一张灰度图**（亮方块 + 高斯亮斑），把上面 `Conv2dMulti` 的权重设成拉普拉斯核，看输出特征图是否勾勒出边缘。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 合成一张 64x64 灰度图：中间一个亮方块，叠加一个高斯亮斑
H = W = 64
img = np.zeros((H, W), dtype='float32')
img[16:48, 16:48] = 200.0                          # 亮方块
yy, xx = np.ogrid[:H, :W]
img += (120 * np.exp(-(((xx - 32) ** 2 + (yy - 32) ** 2) / 400.0))).astype('float32')  # 高斯亮斑
img = np.clip(img, 0, 255)

# 拉普拉斯卷积核
w = torch.tensor([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype=torch.float32)
# 用单输入单输出通道的从零卷积算子，把权重写成拉普拉斯核
conv = Conv2dMulti(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0)
with torch.no_grad():
    conv.weight.copy_(w.reshape(1, 1, 3, 3))

inputs = torch.tensor(img).unsqueeze(0).unsqueeze(0)   # [1, 1, H, W]
outputs = conv(inputs).squeeze().detach()              # [H-2, W-2]
print(f"edge map shape: {tuple(outputs.shape)}  |edge|max={outputs.abs().max():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('synthetic gray image'); axes[0].axis('off')
axes[1].imshow(outputs, cmap='gray'); axes[1].set_title('Laplacian edges'); axes[1].axis('off')
plt.tight_layout(); plt.show()

### 6. 把从零算子拼成 LeNet-5，逐层验证形状

用上面的 `Conv2dMulti` 和 `Pool2D` 搭一个简化版 LeNet-5（卷积层激活统一用 ReLU，汇聚层无参数）。输入 $1\times32\times32$ 灰度图，逐层打印特征图形状，对照书本 §5.3 的张量演化：

`conv1 (1,6,28,28) → pool2 (1,6,14,14) → conv3 (1,16,10,10) → pool4 (1,16,5,5) → conv5 (1,120,1,1) → linear6 (1,84) → linear7 (1,10)`

> 注意：这是用**朴素循环算子**前向，单张图也要算几秒；它只为印证形状与原理，真正训练 LeNet 请用本节最前面基于 `nn.Conv2d` 的版本。

In [ ]:
import torch.nn.functional as F


class Model_LeNet(nn.Module):
    """用从零算子（Conv2dMulti + Pool2D）搭的简化 LeNet-5。"""
    def __init__(self, in_channels, num_classes=10):
        super().__init__()
        self.conv1 = Conv2dMulti(in_channels=in_channels, out_channels=6, kernel_size=5)
        self.pool2 = Pool2D(size=(2, 2), mode='max', stride=2)
        self.conv3 = Conv2dMulti(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.pool4 = Pool2D(size=(2, 2), mode='avg', stride=2)
        self.conv5 = Conv2dMulti(in_channels=16, out_channels=120, kernel_size=5, stride=1)
        self.linear6 = nn.Linear(120, 84)
        self.linear7 = nn.Linear(84, num_classes)

    def forward(self, x):
        output = F.relu(self.conv1(x))           # C1
        output = self.pool2(output)              # S2
        output = F.relu(self.conv3(output))      # C3
        output = self.pool4(output)              # S4
        output = F.relu(self.conv5(output))      # C5
        output = torch.squeeze(output, dim=(2, 3))   # 拉平 [B,C,1,1] -> [B,C]
        output = F.relu(self.linear6(output))    # F6
        output = self.linear7(output)            # F7
        return output


inputs = np.random.randn(1, 1, 32, 32).astype('float32')
model = Model_LeNet(in_channels=1, num_classes=10)
print(model)
x = torch.tensor(inputs)
for name, layer in model.named_children():
    if name == 'linear6':
        x = torch.flatten(x, start_dim=1)        # 进全连接前展平空间维
    x = layer(x)
    print(f"{name:8s} {tuple(x.shape)}")